# Etapa 2 — Pré-processamento (CICDDoS2019 no ClickHouse)

**Regra de ouro deste notebook:** as tabelas originais (`data*`) **nunca são alteradas**. Toda escrita (criar, inserir, apagar)
só é permitida em tabelas com o prefixo `prep_`; qualquer outra tentativa é bloqueada **antes** de chegar ao servidor.

Esta primeira parte é só um **teste de escrita segura**:
1. conecta e lista o que existe no banco (somente leitura);
2. cria uma tabela cópia `prep_teste_copia` (mesma estrutura da original);
3. copia **uma linha** da original para a cópia e imprime o resultado;
4. confirma que a tabela original não mudou.

## 0. Conexão e trava de segurança

In [1]:
import os, re, json, time, http.client, urllib.parse
from pathlib import Path
import pandas as pd

pd.set_option("display.width", 200); pd.set_option("display.max_rows", 200); pd.set_option("display.max_colwidth", 120)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


def load_env(path: Path) -> dict:
    values = {}
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


env_path = REPO_ROOT / ".env"
if not env_path.exists():
    raise FileNotFoundError(f"Arquivo .env não encontrado em: {env_path}")
env = {**load_env(env_path), **os.environ}

CH_HOST, CH_PORT = env["CLICKHOUSE_HOST"], env["CLICKHOUSE_PORT"]
CH_USER, CH_PASSWORD, CH_DATABASE = env["CLICKHOUSE_USER"], env["CLICKHOUSE_PASSWORD"], env["CLICKHOUSE_DATABASE"]
CH_SCHEME = env.get("CLICKHOUSE_SCHEME", "https")
HTTP_TIMEOUT = 1800
print(f"ClickHouse: {CH_HOST}:{CH_PORT}/{CH_DATABASE}  |  usuário: {CH_USER}")

ClickHouse: clickhouse-production-e204.up.railway.app:443/railway  |  usuário: clickhouse


In [2]:
# ---------------- transporte HTTP direto (o mesmo da EDA: conexão reutilizada + reconexão automática) ----------------
class ErroConexao(Exception):
    pass


CONEXAO_ESPERAS = (0, 5, 15, 45, 90)
_conn = None


def _fechar_conexao():
    global _conn
    try:
        if _conn is not None:
            _conn.close()
    except Exception:
        pass
    _conn = None


def _http(query: str) -> pd.DataFrame:
    global _conn
    caminho = "/?" + urllib.parse.urlencode({"database": CH_DATABASE, "default_format": "JSONCompact", "wait_end_of_query": 1})
    cab = {"X-ClickHouse-User": CH_USER, "X-ClickHouse-Key": CH_PASSWORD, "Connection": "keep-alive"}
    ultimo = None
    for espera in CONEXAO_ESPERAS:
        if espera:
            print(f"  conexão instável ({type(ultimo).__name__}); nova tentativa em {espera}s...")
            time.sleep(espera)
        try:
            if _conn is None:
                cls = http.client.HTTPSConnection if CH_SCHEME == "https" else http.client.HTTPConnection
                _conn = cls(CH_HOST, int(CH_PORT), timeout=HTTP_TIMEOUT)
            _conn.request("POST", caminho, body=query.encode("utf-8"), headers=cab)
            resp = _conn.getresponse()
            corpo = resp.read()
        except (OSError, http.client.HTTPException) as e:
            ultimo = e
            _fechar_conexao()
            continue
        if resp.status in (502, 503, 504) or (resp.status >= 500 and not corpo.startswith(b"Code:")):
            ultimo = Exception(f"HTTP {resp.status}")
            _fechar_conexao()
            continue
        if resp.status != 200:
            raise Exception(corpo.decode("utf-8", "replace")[:2000])
        if not corpo.strip():
            return pd.DataFrame()
        dados = json.loads(corpo)
        return pd.DataFrame(dados.get("data", []), columns=[m["name"] for m in dados.get("meta", [])])
    raise ErroConexao(f"Sem conexão com o ClickHouse após {len(CONEXAO_ESPERAS)} tentativas: {ultimo!r}")

In [3]:
# ---------------- TRAVA: só é possível escrever em tabelas prep_* ----------------
PREFIXO_NOVO = "prep_"          # tabelas que ESTE notebook pode criar / alimentar / apagar

_VERBOS_ESCRITA = ("CREATE", "INSERT", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE")
_VERBOS_PROIBIDOS = ("DELETE", "UPDATE", "ATTACH", "DETACH", "EXCHANGE", "GRANT", "REVOKE", "KILL", "SYSTEM", "SET", "USE")
_ALVO = re.compile(r"\b(?:TABLE|INTO|VIEW)\s+(?:IF\s+(?:NOT\s+)?EXISTS\s+)?(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)
_RENAME = re.compile(r"(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)\s+TO\s+(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)


def _tabela_de(ident: str) -> str:
    # 'banco.tabela' ou '`tabela`' -> 'tabela' (e exige que o banco, se informado, seja o atual)
    partes = [p.strip("`") for p in re.findall(r"`[^`]+`|[^.]+", ident)]
    if len(partes) == 2 and partes[0] != CH_DATABASE:
        raise PermissionError(f"Escrita bloqueada: banco '{partes[0]}' diferente do atual '{CH_DATABASE}'.")
    return partes[-1]


def validar_escrita(sql: str, permitir_prefixos=()) -> None:
    # Levanta PermissionError se o comando escreveria fora das tabelas permitidas. Leituras passam sem restrição.
    s = sql.strip().rstrip(";").strip()
    if ";" in s:
        raise PermissionError("Escrita bloqueada: envie uma instrução por vez.")
    verbo = (re.match(r"\s*(\w+)", s).group(1) if re.match(r"\s*(\w+)", s) else "").upper()
    if verbo in _VERBOS_PROIBIDOS:
        raise PermissionError(f"Comando '{verbo}' não é permitido neste notebook.")
    if verbo not in _VERBOS_ESCRITA:
        return                                                    # SELECT / SHOW / DESCRIBE / EXISTS ...
    prefixos = (PREFIXO_NOVO, *permitir_prefixos)
    if verbo == "RENAME":
        nomes = [_tabela_de(x) for par in _RENAME.findall(s) for x in par]
    else:
        nomes = [_tabela_de(x) for x in _ALVO.findall(s)]
    if not nomes:
        raise PermissionError(f"Escrita bloqueada: não identifiquei a tabela alvo em: {s[:80]}...")
    ruins = [n for n in nomes if not n.startswith(prefixos)]
    if ruins:
        raise PermissionError(f"Escrita bloqueada em {ruins}: só é permitido escrever em tabelas com prefixo {prefixos}.")


def executar(sql: str, permitir_prefixos=()) -> pd.DataFrame:
    validar_escrita(sql, permitir_prefixos)
    return _http(sql)


consultar = executar      # leituras também passam pela trava (e ela as libera)

# autoteste da trava (não toca no servidor)
_deve_passar = ["SELECT 1", "SHOW CREATE TABLE data2_portmap", f"CREATE TABLE {PREFIXO_NOVO}x AS data2_portmap",
                f"INSERT INTO {PREFIXO_NOVO}x SELECT * FROM data2_portmap LIMIT 1", f"DROP TABLE IF EXISTS `{PREFIXO_NOVO}x`"]
_deve_bloquear = ["INSERT INTO data2_portmap SELECT 1", "DROP TABLE data2_portmap", "DROP TABLE IF EXISTS data_tftp",
                  "TRUNCATE TABLE data_syn", "ALTER TABLE data_syn DROP COLUMN label", "CREATE TABLE outra AS data_syn",
                  "RENAME TABLE data_syn TO prep_syn", "CREATE DATABASE x", "DELETE FROM data_syn WHERE 1",
                  f"INSERT INTO {PREFIXO_NOVO}x SELECT 1; DROP TABLE data_syn", "SYSTEM DROP MARK CACHE"]
for _s in _deve_passar:
    validar_escrita(_s)
for _s in _deve_bloquear:
    try:
        validar_escrita(_s); raise AssertionError(f"a trava deixou passar: {_s}")
    except PermissionError:
        pass
print(f"Trava de segurança OK: {len(_deve_passar)} comandos permitidos e {len(_deve_bloquear)} bloqueados no autoteste.")

Trava de segurança OK: 5 comandos permitidos e 11 bloqueados no autoteste.


## 1. Inventário do banco (somente leitura)

In [4]:
info = consultar("SELECT version() AS versao, currentDatabase() AS banco, currentUser() AS usuario")
display(info)

tabelas = consultar('''
    SELECT name AS tabela, engine, toString(total_rows) AS linhas, formatReadableSize(total_bytes) AS tamanho
    FROM system.tables WHERE database = currentDatabase() ORDER BY name
''')
tabelas["tipo"] = tabelas["tabela"].map(lambda t: "novo (prep_)" if t.startswith(PREFIXO_NOVO)
                                        else "resto da EDA (eda_tmp_hash_)" if t.startswith("eda_tmp_hash_")
                                        else "original")
display(tabelas)
print(tabelas["tipo"].value_counts().to_string())

,versao,banco,usuario
0,25.8.33.6,railway,clickhouse


,tabela,engine,linhas,tamanho,tipo
0,data2_ldap,MergeTree,2113234,48.41 MiB,original
1,data2_mssql,MergeTree,5775786,232.70 MiB,original
2,data2_netbios,MergeTree,3455899,88.65 MiB,original
3,data2_portmap,MergeTree,191694,5.23 MiB,original
4,data2_syn,MergeTree,4320541,229.29 MiB,original
5,data2_udp,MergeTree,3782206,253.07 MiB,original
6,data2_udplag,MergeTree,725165,39.50 MiB,original
7,data_drdos_dns,MergeTree,5074413,152.10 MiB,original
8,data_drdos_ldap,MergeTree,2181542,55.30 MiB,original
9,data_drdos_mssql,MergeTree,4524498,178.86 MiB,original


tipo
original    18


## 2. Teste: tabela cópia + uma linha

In [5]:
TABELA_ORIGEM = "data2_portmap"                    # a menor das originais (≈ 192 mil linhas); troque se quiser
TABELA_TESTE = PREFIXO_NOVO + "teste_copia"

n_antes = int(consultar(f"SELECT toString(count()) AS n FROM `{TABELA_ORIGEM}`")["n"][0])
print(f"Original `{TABELA_ORIGEM}`: {n_antes:,} linhas (antes do teste)\n")
print("Estrutura da original (SHOW CREATE TABLE):")
print(consultar(f"SHOW CREATE TABLE `{TABELA_ORIGEM}`").iloc[0, 0][:1500], "...")

Original `data2_portmap`: 191,694 linhas (antes do teste)

Estrutura da original (SHOW CREATE TABLE):
CREATE TABLE railway.data2_portmap
(
    `unnamed_0` String CODEC(ZSTD(3)),
    `flow_id` String CODEC(ZSTD(3)),
    `source_ip` String CODEC(ZSTD(3)),
    `source_port` String CODEC(ZSTD(3)),
    `destination_ip` String CODEC(ZSTD(3)),
    `destination_port` String CODEC(ZSTD(3)),
    `protocol` String CODEC(ZSTD(3)),
    `timestamp` String CODEC(ZSTD(3)),
    `flow_duration` String CODEC(ZSTD(3)),
    `total_fwd_packets` String CODEC(ZSTD(3)),
    `total_backward_packets` String CODEC(ZSTD(3)),
    `total_length_of_fwd_packets` String CODEC(ZSTD(3)),
    `total_length_of_bwd_packets` String CODEC(ZSTD(3)),
    `fwd_packet_length_max` String CODEC(ZSTD(3)),
    `fwd_packet_length_min` String CODEC(ZSTD(3)),
    `fwd_packet_length_mean` String CODEC(ZSTD(3)),
    `fwd_packet_length_std` String CODEC(ZSTD(3)),
    `bwd_packet_length_max` String CODEC(ZSTD(3)),
    `bwd_packet_length_min

In [6]:
# 2.1 cria a tabela cópia (mesma estrutura e engine da original; sem dados)
executar(f"DROP TABLE IF EXISTS `{TABELA_TESTE}`")
executar(f"CREATE TABLE `{TABELA_TESTE}` AS `{TABELA_ORIGEM}`")
n_colunas = int(consultar(f"SELECT toString(count()) AS n FROM system.columns WHERE database = currentDatabase() AND table = '{TABELA_TESTE}'")["n"][0])
print(f"✔ Tabela `{TABELA_TESTE}` criada com {n_colunas} colunas.")

✔ Tabela `prep_teste_copia` criada com 88 colunas.


In [7]:
# 2.2 copia UMA linha da original para a cópia
executar(f"INSERT INTO `{TABELA_TESTE}` SELECT * FROM `{TABELA_ORIGEM}` LIMIT 1")

n_copia = int(consultar(f"SELECT toString(count()) AS n FROM `{TABELA_TESTE}`")["n"][0])
print(f"Linhas na cópia: {n_copia}\n")

linha = consultar(f"SELECT * FROM `{TABELA_TESTE}`")
display(linha.T.rename(columns={0: "valor"}))

Linhas na cópia: 1



,valor
unnamed_0,24
flow_id,192.168.50.254-224.0.0.5-0-0-0
source_ip,192.168.50.254
source_port,0
destination_ip,224.0.0.5
destination_port,0
protocol,0
timestamp,2018-11-03 09:18:16.964447
flow_duration,114456999
total_fwd_packets,45


In [8]:
# 2.3 verificações (leves: não lê as 88 colunas da tabela original inteira)
def _lit(v):
    return "'" + str(v).replace("\\", "\\\\").replace("'", "\\'") + "'"


c1, c2 = linha.columns[0], linha.columns[1]            # ex.: unnamed_0 e flow_id
v1, v2 = linha.iloc[0, 0], linha.iloc[0, 1]
h_copia = int(consultar(f"SELECT toString(cityHash64(*)) AS h FROM `{TABELA_TESTE}`")["h"][0])

# 1) filtra a original só pelas 2 colunas-chave (poucas colunas lidas); 2) compara a linha COMPLETA só nas candidatas
existe_na_original = int(consultar(
    f"SELECT toString(count()) AS n FROM (SELECT * FROM `{TABELA_ORIGEM}` WHERE `{c1}` = {_lit(v1)} AND `{c2}` = {_lit(v2)}) "
    f"WHERE cityHash64(*) = {h_copia}")["n"][0])
n_depois = int(consultar(f"SELECT toString(count()) AS n FROM `{TABELA_ORIGEM}`")["n"][0])

print(f"A linha copiada existe na original?            {existe_na_original >= 1}  ({existe_na_original} ocorrência(s))")
print(f"Cópia tem exatamente 1 linha?                  {n_copia == 1}")
print(f"Original inalterada ({n_antes:,} → {n_depois:,})?   {n_antes == n_depois}")
assert existe_na_original >= 1 and n_copia == 1 and n_antes == n_depois, "teste de escrita falhou"
print("\n✔ TESTE CONCLUÍDO: criação, inserção e leitura funcionam e a original não foi alterada.")

A linha copiada existe na original?            True  (1 ocorrência(s))
Cópia tem exatamente 1 linha?                  True
Original inalterada (191,694 → 191,694)?   True

✔ TESTE CONCLUÍDO: criação, inserção e leitura funcionam e a original não foi alterada.


## 3. Limpeza (opcional — só roda se você mandar)

Os flags abaixo vêm como `False` de propósito. Ligue só o que quiser apagar.

In [10]:
APAGAR_TABELA_TESTE = True    # apaga prep_teste_copia
APAGAR_RESTOS_DA_EDA = False     # apaga eda_tmp_hash_* (tabelas auxiliares que a EDA de duplicatas pode ter deixado)

if APAGAR_TABELA_TESTE:
    executar(f"DROP TABLE IF EXISTS `{TABELA_TESTE}`")
    print(f"✔ `{TABELA_TESTE}` apagada.")

restos = tabelas[tabelas["tabela"].str.startswith("eda_tmp_hash_")]["tabela"].tolist()
if APAGAR_RESTOS_DA_EDA:
    for t in restos:
        executar(f"DROP TABLE IF EXISTS `{t}`", permitir_prefixos=("eda_tmp_hash_",))
        print(f"✔ `{t}` apagada.")
elif restos:
    print(f"Há {len(restos)} tabela(s) auxiliar(es) da EDA no banco (eda_tmp_hash_*): ocupam espaço e podem ser apagadas "
          f"com APAGAR_RESTOS_DA_EDA = True.")
else:
    print("Não há restos da EDA (eda_tmp_hash_*) no banco.")

  conexão instável (ConnectionAbortedError); nova tentativa em 5s...
✔ `prep_teste_copia` apagada.
Não há restos da EDA (eda_tmp_hash_*) no banco.
